# 🤖 Model Training — Energy Grid Outage Prediction

Trains a **SynapseML LightGBM** classifier to predict grid outages
based on sensor readings and historical event patterns.

**Target:** `had_outage` (1 = outage/surge/sag event occurred)

**Reads from:** `gold_ml_features`

**Writes to:** `gold_ml_model_metrics`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
features_df = spark.read.table('gold_ml_features')
print(f'Feature rows: {features_df.count():,}')
features_df.groupBy('had_outage').count().show()

In [ ]:
numeric_features = [
    'avg_voltage', 'std_voltage', 'min_voltage', 'max_voltage', 'voltage_range', 'voltage_deviation',
    'avg_frequency', 'std_frequency', 'freq_deviation',
    'avg_power_factor', 'min_power_factor',
    'avg_load', 'max_load',
    'avg_temp', 'max_temp',
    'reading_count',
    'day_of_week', 'month',
]

indexer_region = StringIndexer(inputCol='region', outputCol='region_idx', handleInvalid='keep')
indexed_df = indexer_region.fit(features_df).transform(features_df)

all_features = numeric_features + ['region_idx']

assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df).select('features', col('had_outage').alias('label'))
print(f'Model-ready rows: {model_df.count():,}')
print(f'Feature count: {len(all_features)}')

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,} rows')
print(f'Test:  {test_df.count():,} rows')

In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier

lgbm = LightGBMClassifier(
    featuresCol='features',
    labelCol='label',
    predictionCol='prediction',
    rawPredictionCol='rawPrediction',
    probabilityCol='probability',
    numLeaves=31,
    numIterations=200,
    learningRate=0.05,
    featureFraction=0.8,
    baggingFraction=0.8,
    baggingFreq=5,
    objective='binary',
    isUnbalance=True,
)

model = lgbm.fit(train_df)
print('LightGBM model trained')

In [ ]:
predictions = model.transform(test_df)

auc_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
pr_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderPR')
acc_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')

auc = auc_eval.evaluate(predictions)
auc_pr = pr_eval.evaluate(predictions)
accuracy = acc_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)

print(f'\n=== Model Evaluation ===')
print(f'AUC-ROC:  {auc:.4f}')
print(f'AUC-PR:   {auc_pr:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')

In [ ]:
metrics_data = [
    ('energy-grid', 'outage-prediction', 'LightGBMClassifier',
     len(all_features), train_df.count(), test_df.count(),
     float(auc), float(auc_pr), float(accuracy), float(f1))
]
metrics_df = spark.createDataFrame(
    metrics_data,
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'auc_pr', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())

metrics_df.write.mode('overwrite').format('delta').saveAsTable('gold_ml_model_metrics')
print('Model metrics saved')
metrics_df.show(truncate=False)

In [ ]:
model.save('Files/models/outage_prediction_lgbm')
print('Model saved to Files/models/outage_prediction_lgbm')